# Lab 04 – Decision Tree
## Lab Task 2: Predict the Safety (Acceptability) of a Car

**Goal:** Build a Decision Tree Classifier on the UCI **Car Evaluation** data set to predict how safe/acceptable a car is.

**Before you start:** download the data from the UCI Machine Learning Repository (search "UCI Car Evaluation", page: archive.ics.uci.edu/dataset/19/car+evaluation). You need the file **`car.data`**. It has no header row; the column names are added in the notebook.

**About the target:** the data set rates each car with a final `class` (`unacc`, `acc`, `good`, `vgood`) based on six attributes, one of which is `safety`. The `class` column is the overall rating that the model predicts.

## Step 1: Import libraries

In [ ]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.model_selection import train_test_split
from sklearn import metrics
import matplotlib.pyplot as plt

## Step 2: Load the dataset
Run the cell and choose `car.data` when the file picker appears.

In [ ]:
from google.colab import files
uploaded = files.upload()

col_names = ['buying', 'maint', 'doors', 'persons', 'lug_boot', 'safety', 'class']
df = pd.read_csv("car.data", header=None, names=col_names)
df.head()

## Step 3: Explore the data
| Column | Meaning | Values |
|---|---|---|
| buying | buying price | vhigh, high, med, low |
| maint | maintenance cost | vhigh, high, med, low |
| doors | number of doors | 2, 3, 4, 5more |
| persons | passenger capacity | 2, 4, more |
| lug_boot | luggage boot size | small, med, big |
| safety | estimated safety | low, med, high |
| class | **target**: car rating | unacc, acc, good, vgood |

In [ ]:
print("Shape:", df.shape)
df.info()

In [ ]:
# missing values per column
df.isnull().sum()

In [ ]:
# how many cars fall in each class (target)
print(df['class'].value_counts())

df['class'].value_counts().plot(kind='bar', figsize=(6, 4), title='Class distribution')
plt.show()

There are **no missing values**, but every column is text, so everything must be encoded. The classes are also imbalanced (most cars are `unacc`), which is worth mentioning in your report.

## Step 4: Encode all columns
Machine learning algorithms only work with numbers, so each column is converted with its own `LabelEncoder` (same approach as Activity 2). The encoders are stored so predictions can be decoded back to text later.

In [ ]:
df_encoded = df.copy()
encoders = {}

for col in df_encoded.columns:
    encoders[col] = LabelEncoder()
    df_encoded[col] = encoders[col].fit_transform(df_encoded[col])

df_encoded.head()

In [ ]:
# see what number each category became
for col, enc in encoders.items():
    print(col, "->", dict(zip(enc.classes_, enc.transform(enc.classes_))))

## Step 5: Separate features and target, then split into train and test sets
- **Target (y):** `class`
- **Features (X):** the six car attributes
- 70% training, 30% testing. `stratify=y` keeps the class proportions the same in both sets.

In [ ]:
X = df_encoded.drop('class', axis='columns')
y = df_encoded['class']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=1, stratify=y)

print("Training samples:", X_train.shape[0])
print("Testing samples: ", X_test.shape[0])

## Step 6: Build and train the Decision Tree model (unpruned)

In [ ]:
model = DecisionTreeClassifier(random_state=1)
model.fit(X_train, y_train)

## Step 7: Evaluate the model

In [ ]:
y_pred = model.predict(X_test)

print("Training score:", model.score(X_train, y_train))
print("Test accuracy: ", metrics.accuracy_score(y_test, y_pred))

In [ ]:
class_names = encoders['class'].classes_

print("Confusion matrix:")
print(metrics.confusion_matrix(y_test, y_pred))
print()
print(metrics.classification_report(y_test, y_pred, target_names=class_names))

The unpruned tree usually scores very high on this data set because the ratings follow clear rules based on the attributes. The minority classes (`good`, `vgood`) are the hardest to predict, as the classification report shows.

## Step 8: Prune the tree with `max_depth`
Compare different depths using the `entropy` criterion to find a simpler tree that still performs well.

In [ ]:
results = []
for depth in range(1, 13):
    clf_try = DecisionTreeClassifier(criterion="entropy", max_depth=depth, random_state=1)
    clf_try.fit(X_train, y_train)
    results.append((depth,
                    clf_try.score(X_train, y_train),
                    metrics.accuracy_score(y_test, clf_try.predict(X_test))))

results_df = pd.DataFrame(results, columns=['max_depth', 'train_score', 'test_accuracy'])
results_df

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(results_df['max_depth'], results_df['train_score'], marker='o', label='Training score')
plt.plot(results_df['max_depth'], results_df['test_accuracy'], marker='o', label='Test accuracy')
plt.xlabel('max_depth')
plt.ylabel('Score')
plt.title('Effect of max_depth on accuracy')
plt.legend()
plt.grid(True)
plt.show()

## Step 9: Final model
Using `entropy` with `max_depth=5`. Change the depth if the table above shows a better choice.

In [ ]:
clf = DecisionTreeClassifier(criterion="entropy", max_depth=5, random_state=1)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)
print("Final model training score:", clf.score(X_train, y_train))
print("Final model test accuracy: ", metrics.accuracy_score(y_test, y_pred))
print()
print(metrics.classification_report(y_test, y_pred, target_names=class_names, zero_division=0))

## Step 10: Visualize the decision tree

In [ ]:
plt.figure(figsize=(24, 10))
plot_tree(clf,
          feature_names=X.columns,
          class_names=class_names,
          filled=True, rounded=True, fontsize=8)
plt.show()

## Step 11: Which features matter most?

In [ ]:
importance = pd.Series(clf.feature_importances_, index=X.columns).sort_values(ascending=False)
print(importance)

importance.plot(kind='bar', figsize=(7, 4), title='Feature importance')
plt.show()

## Step 12: Predict the rating of new cars
Values are text, converted with the stored encoders, and the predicted number is converted back to a class name.

In [ ]:
def predict_car(buying, maint, doors, persons, lug_boot, safety):
    row = pd.DataFrame([[
        encoders['buying'].transform([buying])[0],
        encoders['maint'].transform([maint])[0],
        encoders['doors'].transform([doors])[0],
        encoders['persons'].transform([persons])[0],
        encoders['lug_boot'].transform([lug_boot])[0],
        encoders['safety'].transform([safety])[0],
    ]], columns=X.columns)
    pred = clf.predict(row)
    return encoders['class'].inverse_transform(pred)[0]

# Car 1: cheap, low maintenance, 4 doors, 4 persons, big boot, high safety
print("Car 1:", predict_car('low', 'low', '4', '4', 'big', 'high'))

# Car 2: same car but with low safety
print("Car 2:", predict_car('low', 'low', '4', '4', 'big', 'low'))

# Car 3: expensive, high maintenance, 2 doors, 2 persons, small boot, medium safety
print("Car 3:", predict_car('vhigh', 'vhigh', '2', '2', 'small', 'med'))

Car 2 shows the effect of safety. A car with `low` safety is rated unacceptable (`unacc`) no matter how good its other attributes are.

## Conclusion
- A decision tree classifier was built on the UCI Car Evaluation data set to predict a car's rating.
- All attributes were categorical, so label encoding was applied to every column.
- The model was evaluated with accuracy, a confusion matrix and a classification report.
- Pruning with `max_depth` and the `entropy` criterion produced a simpler tree that is easier to interpret.
- `safety` and `persons` are the most influential features: cars with low safety or capacity for only 2 persons are almost always rated unacceptable.